# 02 Theory Checks

Executable notebook for hypotheses / theory checks. This is not model training and not final model decision making.

Each block states the hypothesis, data used, risk / limitation, and boundary. Target-based checks use `train.csv` only. `test.csv` is used only for structural and alignment checks without `Survived`.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "train.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

def extract_title(name):
    match = re.search(r",\s*([^.]*)\.", str(name))
    if not match:
        return "Unknown"
    title = match.group(1).strip()
    if title in {"Mlle", "Ms"}:
        return "Miss"
    if title == "Mme":
        return "Mrs"
    if title in {"Mr", "Mrs", "Miss", "Master"}:
        return title
    return "Rare"

def ticket_prefix(ticket):
    if pd.isna(ticket):
        return "NONE"
    text = str(ticket).upper().strip()
    parts = re.split(r"\s+", text)
    prefix_parts = [re.sub(r"[^A-Z0-9]", "", part) for part in parts[:-1]]
    prefix_parts = [part for part in prefix_parts if part and not part.isdigit()]
    if not prefix_parts:
        first = re.sub(r"[^A-Z]", "", parts[0])
        return first if first else "NONE"
    return "_".join(prefix_parts)

def bucket_family_size(size):
    if pd.isna(size):
        return "unknown"
    value = int(size)
    if value == 1:
        return "alone"
    if value <= 4:
        return "small"
    if value <= 6:
        return "medium"
    return "large"

def add_clean_eda_features(frame):
    out = frame.copy()
    out["Title"] = out["Name"].map(extract_title)
    out["AgeMissing"] = out["Age"].isna().astype(int)
    out["CabinKnown"] = out["Cabin"].notna().astype(int)
    out["FamilySize"] = out["SibSp"] + out["Parch"] + 1
    out["FareLog"] = np.log1p(out["Fare"])
    out["FarePerPerson"] = out["Fare"] / out["FamilySize"].replace(0, np.nan)
    out["FarePerPersonLog"] = np.log1p(out["FarePerPerson"])
    out["TicketPrefix"] = out["Ticket"].map(ticket_prefix)
    out["FamilySizeBucket"] = out["FamilySize"].map(bucket_family_size)
    return out

def add_rank_bins(frame, columns):
    out = frame.copy()
    labels = ["q1_low", "q2_mid", "q3_high", "q4_very_high"]
    for column in columns:
        ranked = out[column].rank(method="first")
        out[f"{column}Bin"] = pd.qcut(ranked, q=4, labels=labels).astype("object")
        out.loc[out[column].isna(), f"{column}Bin"] = "missing"
    return out

def rate_table(frame, group_cols):
    return (
        frame.groupby(group_cols, dropna=False)["Survived"]
        .agg(rows="count", survived="sum", survival_rate="mean")
        .reset_index()
        .sort_values(group_cols)
    )

train = add_rank_bins(add_clean_eda_features(train_raw), ["Fare", "FareLog", "FarePerPerson", "FarePerPersonLog"])
test = add_rank_bins(add_clean_eda_features(test_raw), ["Fare", "FareLog", "FarePerPerson", "FarePerPersonLog"])

display(pd.DataFrame({"file": ["train.csv", "test.csv"], "rows": [len(train), len(test)]}))

,file,rows
0,train.csv,891
1,test.csv,418


## Rule Sanity Checks

- Hypothesis / theory check: simple outcome-aware rules provide train-only sanity checks for known Titanic structure.
- Data used: `train.csv` only.
- Risk / limitation: train accuracy is descriptive and can overfit; these checks are not selected solution, final pipeline, CV, or submission logic.
- Boundary: this is one compact sensitivity summary, not a baseline ladder.
- Age-threshold sensitivity is handled in the dedicated Age section below, after `Title` context.

In [2]:
rule_checks = []

all_died_pred = pd.Series(0, index=train.index)
gender_rule_pred = train["Sex"].eq("female").astype(int)

rule_checks.append(
    {
        "check": "all-died train accuracy",
        "threshold": "n/a",
        "train_accuracy": all_died_pred.eq(train["Survived"]).mean(),
    }
)
rule_checks.append(
    {
        "check": "gender rule train accuracy",
        "threshold": "n/a",
        "train_accuracy": gender_rule_pred.eq(train["Survived"]).mean(),
    }
)

rule_checks = pd.DataFrame(rule_checks)
rule_checks["train_accuracy"] = rule_checks["train_accuracy"].round(4)
display(rule_checks)

,check,threshold,train_accuracy
0,all-died train accuracy,n/a,0.6162
1,gender rule train accuracy,n/a,0.7868


Interpretation: In train, the all-died and gender rules provide a compact sanity reference only. Age-threshold behavior is reviewed separately in the Age section so the child signal is not mixed into the generic rule block. Boundary: this is an EDA observation, not a final modeling decision.

## Title Signal Context

- Hypothesis / theory check: parsed `Title` captures passenger social/family structure and can help interpret missing age and child-male rows.
- Data used: `train.csv` only for survival rates.
- Risk / limitation: broad `Title` already showed public-transfer risk in later checkpoints; this block is context for narrow Age logic, not a feature promotion.
- Boundary: this is not a final model decision.

In [3]:
display(rate_table(train, ["Title"]).rename(columns={"rows": "count", "survived": "survived_count"}).round(4))
display(rate_table(train, ["Title", "Sex", "Pclass"]).rename(columns={"rows": "count", "survived": "survived_count"}).round(4))

,Title,count,survived_count,survival_rate
0,Master,40,23,0.5750
1,Miss,185,130,0.7027
2,Mr,517,81,0.1567
3,Mrs,126,100,0.7937
4,Rare,23,8,0.3478


,Title,Sex,Pclass,count,survived_count,survival_rate
0,Master,male,1,3,3,1.0000
1,Master,male,2,9,9,1.0000
2,Master,male,3,28,11,0.3929
3,Miss,female,1,48,46,0.9583
4,Miss,female,2,35,33,0.9429
5,Miss,female,3,102,51,0.5000
6,Mr,male,1,107,37,0.3458
7,Mr,male,2,91,8,0.0879
8,Mr,male,3,319,36,0.1129
9,Mrs,female,1,43,42,0.9767


Interpretation: `Title` is kept here as structural EDA context. Broad `Title` is not being returned as a full-strength feature in this step; the Age section below only considers `Master` as a narrow missing-age child-male fallback. Boundary: this is an EDA observation, not a final modeling decision.

## Age / AgeMissing / AgeBucket theory checks

This section collects Age-related EDA in one place after the `Title` context block. It replaces the earlier scattered `AgeMissing` and child-threshold checks.

### 1. Age missingness overview

Goal: review missingness by `Sex`, `Pclass`, and `Title` before proposing any AgeBucket branch.

In [4]:
def age_survival_table(frame, group_cols):
    table = (
        frame.groupby(group_cols, dropna=False)["Survived"]
        .agg(count="count", survived_count="sum", survival_rate="mean")
        .reset_index()
    )
    table["survival_rate"] = table["survival_rate"].round(4)
    return table.sort_values(group_cols).reset_index(drop=True)

age_frame = train.copy()
age_frame["AgeStatus"] = np.where(age_frame["Age"].isna(), "Age missing", "Age present")

display(age_survival_table(age_frame, ["AgeStatus"]))
display(age_survival_table(age_frame, ["AgeMissing", "Sex"]))
display(age_survival_table(age_frame, ["AgeMissing", "Sex", "Pclass"]))
display(age_survival_table(age_frame, ["AgeMissing", "Title"]))

,AgeStatus,count,survived_count,survival_rate
0,Age missing,177,52,0.2938
1,Age present,714,290,0.4062


,AgeMissing,Sex,count,survived_count,survival_rate
0,0,female,261,197,0.7548
1,0,male,453,93,0.2053
2,1,female,53,36,0.6792
3,1,male,124,16,0.1290


,AgeMissing,Sex,Pclass,count,survived_count,survival_rate
0,0,female,1,85,82,0.9647
1,0,female,2,74,68,0.9189
2,0,female,3,102,47,0.4608
3,0,male,1,101,40,0.3960
4,0,male,2,99,15,0.1515
5,0,male,3,253,38,0.1502
6,1,female,1,9,9,1.0000
7,1,female,2,2,2,1.0000
8,1,female,3,42,25,0.5952
9,1,male,1,21,5,0.2381


,AgeMissing,Title,count,survived_count,survival_rate
0,0,Master,36,21,0.5833
1,0,Miss,149,108,0.7248
2,0,Mr,398,67,0.1683
3,0,Mrs,109,86,0.7890
4,0,Rare,22,8,0.3636
5,1,Master,4,2,0.5000
6,1,Miss,36,22,0.6111
7,1,Mr,119,14,0.1176
8,1,Mrs,17,14,0.8235
9,1,Rare,1,0,0.0000


Interpretation: Missing `Age` is structured across `Sex`, `Pclass`, and `Title`, so it should not simply be added on top of median-imputed raw `Age` as the main next step. The next controlled idea is a separate AgeBucket branch where raw `Age` is removed from the candidate feature set and age information is passed through a bucket. Boundary: this is an EDA observation, not a final modeling decision.

### 2. Child threshold check by Sex

Thresholds are checked before choosing a candidate. Adult-or-unknown includes rows of the same sex that are not known-age children at the threshold.

In [5]:
child_threshold_rows = []
for threshold in [12, 14, 16, 18]:
    child_flag = train["Age"].notna() & (train["Age"] < threshold)
    for sex in ["male", "female"]:
        for is_child, group in [(True, f"{sex}_child"), (False, f"{sex}_adult_or_unknown")]:
            mask = train["Sex"].eq(sex) & (child_flag if is_child else ~child_flag)
            subset = train.loc[mask]
            child_threshold_rows.append(
                {
                    "threshold": threshold,
                    "group": group,
                    "count": len(subset),
                    "survived_count": int(subset["Survived"].sum()),
                    "survival_rate": subset["Survived"].mean(),
                }
            )

child_threshold_table = pd.DataFrame(child_threshold_rows)
child_threshold_table["survival_rate"] = child_threshold_table["survival_rate"].round(4)
display(child_threshold_table)

,threshold,group,count,survived_count,survival_rate
0,12,male_child,36,20,0.5556
1,12,male_adult_or_unknown,541,89,0.1645
2,12,female_child,32,19,0.5938
3,12,female_adult_or_unknown,282,214,0.7589
4,14,male_child,37,21,0.5676
5,14,male_adult_or_unknown,540,88,0.1630
6,14,female_child,34,21,0.6176
7,14,female_adult_or_unknown,280,212,0.7571
8,16,male_child,40,21,0.5250
9,16,male_adult_or_unknown,537,88,0.1639


Interpretation: The threshold table is used to compare 12, 14, 16, and 18 directly rather than choosing a child cutoff manually. The main check is where male-child survival starts to deteriorate, especially whether 12/14 look cleaner than 16 and whether 18 weakens the signal. Boundary: this is an EDA observation, not a final modeling decision.

### 3. Sex x AgeBand survival table

Known-age rows are bucketed into fixed bands to show where survival changes by sex.

In [6]:
age_bins = [0, 5, 11, 13, 15, 17, 24, 34, 44, 54, 64, 120]
age_labels = [
    "0-5",
    "6-11",
    "12-13",
    "14-15",
    "16-17",
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65+",
]

ageband_frame = train.loc[train["Age"].notna()].copy()
ageband_frame["AgeBand"] = pd.cut(
    ageband_frame["Age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True,
    right=True,
)
ageband_table = age_survival_table(ageband_frame, ["Sex", "AgeBand"])
display(ageband_table)
display(ageband_table[ageband_table["Sex"].eq("male")].reset_index(drop=True))
display(ageband_table[ageband_table["Sex"].eq("female")].reset_index(drop=True))

,Sex,AgeBand,count,survived_count,survival_rate
0,female,0-5,21,16,0.7619
1,female,6-11,11,3,0.2727
2,female,12-13,2,2,1.0000
3,female,14-15,9,7,0.7778
4,female,16-17,12,10,0.8333
5,female,18-24,62,47,0.7581
6,female,25-34,63,47,0.7460
7,female,35-44,45,36,0.8000
8,female,45-54,26,20,0.7692
9,female,55-64,10,9,0.9000


,Sex,AgeBand,count,survived_count,survival_rate
0,male,0-5,23,15,0.6522
1,male,6-11,13,5,0.3846
2,male,12-13,1,1,1.0000
3,male,14-15,3,0,0.0000
4,male,16-17,18,2,0.1111
5,male,18-24,102,10,0.0980
6,male,25-34,138,31,0.2246
7,male,35-44,76,15,0.1974
8,male,45-54,47,10,0.2128
9,male,55-64,21,3,0.1429


,Sex,AgeBand,count,survived_count,survival_rate
0,female,0-5,21,16,0.7619
1,female,6-11,11,3,0.2727
2,female,12-13,2,2,1.0000
3,female,14-15,9,7,0.7778
4,female,16-17,12,10,0.8333
5,female,18-24,62,47,0.7581
6,female,25-34,63,47,0.7460
7,female,35-44,45,36,0.8000
8,female,45-54,26,20,0.7692
9,female,55-64,10,9,0.9000


Interpretation: The Sex x AgeBand table separates male and female age patterns so any child/adult/old breakpoints can be proposed from visible train-side rates rather than assumed. Boundary: this is an EDA observation, not a final modeling decision.

### 4. Old threshold candidates by Sex

Old thresholds are checked separately by sex on known-age rows only. Missing-age rows are handled before old-threshold logic in the proposed mapping.

In [7]:
old_threshold_rows = []
old_thresholds = [45, 50, 55, 60, 65]
for sex in ["female", "male"]:
    sex_known = train["Sex"].eq(sex) & train["Age"].notna()
    for threshold in old_thresholds:
        old_mask = sex_known & train["Age"].ge(threshold)
        non_old_mask = sex_known & train["Age"].lt(threshold)
        old_subset = train.loc[old_mask]
        non_old_subset = train.loc[non_old_mask]
        old_rate = old_subset["Survived"].mean()
        non_old_rate = non_old_subset["Survived"].mean()
        old_threshold_rows.append(
            {
                "Sex": sex,
                "threshold": threshold,
                "old_count": len(old_subset),
                "old_survived_count": int(old_subset["Survived"].sum()),
                "old_survival_rate": old_rate,
                "non_old_count": len(non_old_subset),
                "non_old_survival_rate": non_old_rate,
                "delta": old_rate - non_old_rate,
            }
        )

old_threshold_table = pd.DataFrame(old_threshold_rows)
for column in ["old_survival_rate", "non_old_survival_rate", "delta"]:
    old_threshold_table[column] = old_threshold_table[column].round(4)
display(old_threshold_table)

,Sex,threshold,old_count,old_survived_count,old_survival_rate,non_old_count,non_old_survival_rate,delta
0,female,45,36,29,0.8056,225,0.7467,0.0589
1,female,50,22,20,0.9091,239,0.7406,0.1685
2,female,55,10,9,0.9000,251,0.7490,0.1510
3,female,60,4,4,1.0000,257,0.7510,0.2490
4,female,65,0,0,NaN,261,0.7548,NaN
5,male,45,79,14,0.1772,374,0.2112,-0.0340
6,male,50,52,7,0.1346,401,0.2145,-0.0798
7,male,55,32,4,0.1250,421,0.2114,-0.0864
8,male,60,22,3,0.1364,431,0.2088,-0.0725
9,male,65,11,1,0.0909,442,0.2081,-0.1172


Interpretation: Old-threshold candidates are sex-specific and data-driven here. If counts are small or there is no stable breakpoint, the correct conclusion is to defer or reject a general old threshold rather than force one. Boundary: this is an EDA observation, not a final modeling decision.

### 5. Master as missing-age child-male fallback

This check tests only whether `Title == "Master"` can help route missing-age male rows. It does not restore broad `Title`.

In [8]:
master = train[train["Title"].eq("Master")].copy()
master_age_status = master.assign(
    MasterAgeStatus=np.where(master["Age"].isna(), "Master Age missing", "Master Age present")
)
display(age_survival_table(master_age_status, ["MasterAgeStatus"]))

master_known_age = master.loc[master["Age"].notna(), "Age"]
display(
    pd.DataFrame(
        [
            {
                "known_age_count": int(master_known_age.count()),
                "known_age_min": master_known_age.min(),
                "known_age_median": master_known_age.median(),
                "known_age_max": master_known_age.max(),
            }
        ]
    )
)

display(
    master.loc[
        master["Age"].isna(),
        ["PassengerId", "Name", "Sex", "Pclass", "Age", "SibSp", "Parch", "Fare", "Survived"],
    ]
)

,MasterAgeStatus,count,survived_count,survival_rate
0,Master Age missing,4,2,0.5000
1,Master Age present,36,21,0.5833


,known_age_count,known_age_min,known_age_median,known_age_max
0,36,0.42,3.5,12.0


,PassengerId,Name,Sex,Pclass,Age,SibSp,Parch,Fare,Survived
65,66,"Moubarek, Master. Gerios",male,3,NaN,1,1,15.2458,1
159,160,"Sage, Master. Thomas Henry",male,3,NaN,8,2,69.5500,0
176,177,"Lefebre, Master. Henry Forbes",male,3,NaN,3,1,25.4667,0
709,710,"Moubarek, Master. Halim Gonios (""William George"")",male,3,NaN,1,1,15.2458,1


Interpretation: Broad `Title` is not returned. `Master` remains eligible only as a narrow Title-derived fallback for missing-age male rows:

```python
if Sex == "male" and Title == "Master" and Age is missing:
    AgeBucket = "ChildMale"
```

Do not use `Master` to rewrite known-Age rows at this stage. Boundary: this is an EDA observation, not a final modeling decision.

### 6. Proposed AgeBucket mapping summary

Candidate logic for a later controlled feature check only:

```python
if Age is missing and Sex == "male" and Title == "Master":
    AgeBucket = "ChildMale"
elif Age is missing and Sex == "female":
    AgeBucket = "AgeMissingFemale"
elif Age is missing and Sex == "male":
    AgeBucket = "AgeMissingMale"
elif Sex == "female" and Age < child_threshold:
    AgeBucket = "ChildFemale"
elif Sex == "male" and Age < child_threshold:
    AgeBucket = "ChildMale"
elif Sex == "female" and Age >= old_female_threshold:
    AgeBucket = "OldFemale"
elif Sex == "male" and Age >= old_male_threshold:
    AgeBucket = "OldMale"
elif Sex == "female":
    AgeBucket = "AdultFemale"
else:
    AgeBucket = "AdultMale"
```

- `child_threshold` is chosen after child threshold EDA.
- `old_female_threshold` is chosen after female old threshold EDA.
- `old_male_threshold` is chosen after male old threshold EDA.
- This is not implemented in `features.py` or preprocessing in this step.

### 7. What not to do

- do not change `scripts/preprocessing.py`;
- do not change `scripts/features.py`;
- do not run model CV;
- do not create submission;
- do not use `gender_submission.csv` as truth;
- do not use test target, it is not available;
- do not return broad `Title`;
- do not add `Mrs` / `Miss` buckets in this step;
- do not do Surname/family inference in this step.

### Current status after frozen checks

- Age EDA remains useful as historical analysis.
- AgeBucket v1 and AgeMissing handling were already checked through frozen public checkpoints.
- They did not beat the current clean leader.
- Therefore AgeBucket / AgeMissing are closed for the current path.
- Do not reopen Age from this notebook without a new explicit hypothesis.
- This Age block should be read as historical EDA, not as an active next-feature recommendation.


## Fare / Cabin overlap EDA

This unified section keeps the previously executed Fare-derived and socioeconomic-overlap checks first, then adds the missing Fare/Cabin overlap audit. It remains descriptive EDA only.


### Fare-Derived Signals

- Hypothesis / theory check: `Fare`, `FareLog`, `FarePerPerson`, and `FarePerPersonLog` may capture status and within-class price variation.
- Data used: `train.csv` with train-only `Survived` rates for EDA.
- Risk / limitation: fare signals overlap with `Pclass`, family size, and cabin availability.
- Boundary: this is not a final model decision.

In [9]:
fare_features = ["Fare", "FareLog", "FarePerPerson", "FarePerPersonLog"]
display(train[fare_features].describe().T.round(4))
display(rate_table(train, ["FareBin"]).round(4))
display(rate_table(train, ["FareBin", "Pclass"]).round(4))
display(rate_table(train, ["FarePerPersonBin"]).round(4))
display(rate_table(train, ["FarePerPersonBin", "Pclass"]).round(4))

,count,mean,std,min,25%,50%,75%,max
Fare,891.0,32.2042,49.6934,0.0,7.9104,14.4542,31.0000,512.3292
FareLog,891.0,2.9622,0.9690,0.0,2.1872,2.7379,3.4657,6.2409
FarePerPerson,891.0,19.9164,35.8413,0.0,7.2500,8.3000,23.6667,512.3292
FarePerPersonLog,891.0,2.5650,0.8577,0.0,2.1102,2.2300,3.2055,6.2409


,FareBin,rows,survived,survival_rate
0,q1_low,223,44,0.1973
1,q2_mid,223,67,0.3004
2,q3_high,222,101,0.4550
3,q4_very_high,223,130,0.5830


,FareBin,Pclass,rows,survived,survival_rate
0,q1_low,1,6,0,0.0000
1,q1_low,2,6,0,0.0000
2,q1_low,3,211,44,0.2085
3,q2_mid,2,86,33,0.3837
4,q2_mid,3,137,34,0.2482
5,q3_high,1,50,26,0.5200
6,q3_high,2,70,42,0.6000
7,q3_high,3,102,33,0.3235
8,q4_very_high,1,160,110,0.6875
9,q4_very_high,2,22,12,0.5455


,FarePerPersonBin,rows,survived,survival_rate
0,q1_low,223,60,0.2691
1,q2_mid,223,56,0.2511
2,q3_high,222,91,0.4099
3,q4_very_high,223,135,0.6054


,FarePerPersonBin,Pclass,rows,survived,survival_rate
0,q1_low,1,6,0,0.0000
1,q1_low,2,21,12,0.5714
2,q1_low,3,196,48,0.2449
3,q2_mid,2,1,1,1.0000
4,q2_mid,3,222,55,0.2477
5,q3_high,1,9,8,0.8889
6,q3_high,2,149,72,0.4832
7,q3_high,3,64,11,0.1719
8,q4_very_high,1,201,128,0.6368
9,q4_very_high,2,13,2,0.1538


Interpretation: The displayed bins show higher train survival in the top `FareBin` (0.5830) than in the low `FareBin` (0.1973), and the top `FarePerPersonBin` is 0.6054 while the lower bins are 0.2691, 0.2511, and 0.4099. Within `Pclass` splits, the shown rates are not monotonic in every small subgroup. Boundary: this is an EDA observation, not a final modeling decision.

### Socioeconomic Overlap: Pclass / CabinKnown / FareLog / FarePerPersonLog

- Hypothesis / theory check: `Pclass`, `CabinKnown`, `FareLog`, and `FarePerPersonLog` overlap but are not identical.
- Data used: `train.csv`; `Survived` appears only in train-side summary and correlation checks.
- Risk / limitation: correlation is descriptive and does not prove independent validation value.
- Boundary: this is not a final model decision.

In [10]:
socio_summary = (
    train.groupby(["Pclass", "CabinKnown"], dropna=False)
    .agg(
        rows=("PassengerId", "count"),
        survival_rate=("Survived", "mean"),
        fare_log_median=("FareLog", "median"),
        fare_per_person_log_median=("FarePerPersonLog", "median"),
        fare_log_mean=("FareLog", "mean"),
        fare_per_person_log_mean=("FarePerPersonLog", "mean"),
    )
    .reset_index()
)
display(socio_summary.round(4))

corr_columns = ["Pclass", "CabinKnown", "FareLog", "FarePerPersonLog", "Survived"]
display(train[corr_columns].corr(numeric_only=True).round(4))

,Pclass,CabinKnown,rows,survival_rate,fare_log_median,fare_per_person_log_median,fare_log_mean,fare_per_person_log_mean
0,1,0,40,0.4750,3.8219,3.4610,3.8531,3.6800
1,1,1,176,0.6648,4.2332,3.5787,4.1534,3.6634
2,2,0,168,0.4405,2.7740,2.6391,2.8908,2.4843
3,2,1,16,0.8125,2.6391,2.4423,2.8442,2.4873
4,3,0,479,0.2359,2.2028,2.1691,2.4912,2.1133
5,3,1,12,0.5000,2.4391,2.0569,2.4852,2.0027


,Pclass,CabinKnown,FareLog,FarePerPersonLog,Survived
Pclass,1.0000,-0.7255,-0.6610,-0.7224,-0.3385
CabinKnown,-0.7255,1.0000,0.5572,0.5775,0.3169
FareLog,-0.6610,0.5572,1.0000,0.8232,0.3299
FarePerPersonLog,-0.7224,0.5775,0.8232,1.0000,0.2988
Survived,-0.3385,0.3169,0.3299,0.2988,1.0000


Interpretation: The grouped table shows higher train survival for `CabinKnown` rows than cabin-unknown rows within each `Pclass`. The correlation table shows `FareLog` and `FarePerPersonLog` are strongly related to each other (0.8232), both are negative with `Pclass`, and both are positive with `CabinKnown` and `Survived`. Boundary: this is an EDA observation, not a final modeling decision.

### Fare feature summary and missingness

- Hypothesis / theory check: `Fare` and `FareLog` may carry socioeconomic signal, but missingness and train/test structure must be checked before any feature decision.
- Data used: `train.csv` and `test.csv`; no test target is used.
- Boundary: `FarePerPerson` and `FarePerPersonLog` remain only in the preserved old check above and are not promoted here.


In [11]:
fare_train = train.copy()
fare_test = test.copy()
fare_datasets = [('train', fare_train), ('test', fare_test)]

fare_missingness = []
farelog_check = []
for dataset_name, frame in fare_datasets:
    expected_farelog = np.log1p(frame['Fare'])
    max_abs_farelog_error = (frame['FareLog'] - expected_farelog).abs().max(skipna=True)
    fare_missingness.append(
        {
            'dataset': dataset_name,
            'rows': len(frame),
            'fare_missing': int(frame['Fare'].isna().sum()),
            'fare_missing_rate': frame['Fare'].isna().mean(),
        }
    )
    farelog_check.append(
        {
            'dataset': dataset_name,
            'farelog_missing': int(frame['FareLog'].isna().sum()),
            'farelog_missing_matches_fare_missing': frame['FareLog'].isna().equals(frame['Fare'].isna()),
            'farelog_min': frame['FareLog'].min(skipna=True),
            'farelog_median': frame['FareLog'].median(skipna=True),
            'farelog_max': frame['FareLog'].max(skipna=True),
            'max_abs_farelog_error': max_abs_farelog_error,
        }
    )

display(pd.DataFrame(fare_missingness).round(4))
display(pd.DataFrame(farelog_check).round(6))


,dataset,rows,fare_missing,fare_missing_rate
0,train,891,0,0.0000
1,test,418,1,0.0024


,dataset,farelog_missing,farelog_missing_matches_fare_missing,farelog_min,farelog_median,farelog_max,max_abs_farelog_error
0,train,0,True,0.0,2.737881,6.240917,0.0
1,test,1,True,0.0,2.737881,6.240917,0.0


### Fare / FareLog vs Pclass

- Hypothesis / theory check: fare distributions should be read against `Pclass` before treating fare as marginal signal.
- Data used: `train.csv` and `test.csv`; no target is needed for this structural alignment check.


In [12]:
def grouped_quantiles(frame, group_col, value_col, dataset_name):
    quantiles = (
        frame.groupby(group_col, dropna=False)[value_col]
        .quantile([0, 0.25, 0.5, 0.75, 0.9, 1.0])
        .unstack()
        .rename(columns={0: 'q0', 0.25: 'q25', 0.5: 'q50', 0.75: 'q75', 0.9: 'q90', 1.0: 'q100'})
        .reset_index()
    )
    quantiles.insert(0, 'dataset', dataset_name)
    return quantiles

pclass_counts = []
for dataset_name, frame in fare_datasets:
    counts = frame.groupby('Pclass', dropna=False).size().reset_index(name='count')
    counts.insert(0, 'dataset', dataset_name)
    pclass_counts.append(counts)

display(pd.concat(pclass_counts, ignore_index=True))
for value_col in ['Fare', 'FareLog']:
    display(
        pd.concat(
            [grouped_quantiles(frame, 'Pclass', value_col, dataset_name) for dataset_name, frame in fare_datasets],
            ignore_index=True,
        ).round(4)
    )


,dataset,Pclass,count
0,train,1,216
1,train,2,184
2,train,3,491
3,test,1,107
4,test,2,93
5,test,3,218


,dataset,Pclass,q0,q25,q50,q75,q90,q100
0,train,1,0.0000,30.924,60.2875,93.5,159.1646,512.3292
1,train,2,0.0000,13.000,14.2500,26.0,33.0000,73.5000
2,train,3,0.0000,7.750,8.0500,15.5,27.9000,69.5500
3,test,1,0.0000,30.100,60.0000,134.5,221.7792,512.3292
4,test,2,9.6875,13.000,15.7500,26.0,36.9534,73.5000
5,test,3,3.1708,7.750,7.8958,14.4,22.0250,69.5500


,dataset,Pclass,q0,q25,q50,q75,q90,q100
0,train,1,0.0000,3.4633,4.1155,4.5486,5.0756,6.2409
1,train,2,0.0000,2.6391,2.7244,3.2958,3.5264,4.3108
2,train,3,0.0000,2.1691,2.2028,2.8034,3.3638,4.2563
3,test,1,0.0000,3.4371,4.1109,4.9090,5.4062,6.2409
4,test,2,2.3691,2.6391,2.8184,3.2958,3.6364,4.3108
5,test,3,1.4281,2.1691,2.1856,2.7344,3.1366,4.2563


### FareLog bins train/test alignment

- Hypothesis / theory check: `FareLog` bins should be stable when bin edges are learned from train and applied to test.
- Data used: train-side bin edges; train/test counts and percentages only.


In [13]:
farelog_nonmissing = fare_train['FareLog'].dropna()
_, farelog_edges = pd.qcut(farelog_nonmissing, q=4, retbins=True, duplicates='drop')
farelog_edges = np.unique(farelog_edges)
farelog_cut_edges = farelog_edges.copy()
farelog_cut_edges[0] = -np.inf
farelog_cut_edges[-1] = np.inf

base_bin_names = ['low', 'mid_low', 'mid_high', 'high', 'very_high', 'extreme']
farelog_bin_labels = []
for i in range(len(farelog_cut_edges) - 1):
    bin_name = base_bin_names[i] if i < len(base_bin_names) else 'bin'
    farelog_bin_labels.append('q%s_%s' % (i + 1, bin_name))
farelog_bin_order = farelog_bin_labels + ['missing']

def add_train_farelog_bin(frame):
    out = frame.copy()
    out['FareLogBinTrain'] = pd.cut(
        out['FareLog'],
        bins=farelog_cut_edges,
        labels=farelog_bin_labels,
        include_lowest=True,
    ).astype('object')
    out.loc[out['FareLog'].isna(), 'FareLogBinTrain'] = 'missing'
    return out

def farelog_bin_distribution(frame, dataset_name):
    counts = frame['FareLogBinTrain'].value_counts(dropna=False).reindex(farelog_bin_order, fill_value=0)
    result = counts.rename_axis('FareLogBinTrain').reset_index(name='count')
    result.insert(0, 'dataset', dataset_name)
    result['pct'] = result['count'] / len(frame)
    return result

fare_train = add_train_farelog_bin(fare_train)
fare_test = add_train_farelog_bin(fare_test)
fare_datasets = [('train', fare_train), ('test', fare_test)]

farelog_edges_table = pd.DataFrame(
    {
        'FareLogBinTrain': farelog_bin_labels,
        'left_edge': farelog_cut_edges[:-1],
        'right_edge': farelog_cut_edges[1:],
    }
)
display(farelog_edges_table.round(4))
display(pd.concat([farelog_bin_distribution(frame, dataset_name) for dataset_name, frame in fare_datasets], ignore_index=True).round(4))


,FareLogBinTrain,left_edge,right_edge
0,q1_low,-inf,2.1872
1,q2_mid_low,2.1872,2.7379
2,q3_mid_high,2.7379,3.4657
3,q4_high,3.4657,inf


,dataset,FareLogBinTrain,count,pct
0,train,q1_low,223,0.2503
1,train,q2_mid_low,224,0.2514
2,train,q3_mid_high,222,0.2492
3,train,q4_high,222,0.2492
4,train,missing,0,0.0000
5,test,q1_low,114,0.2727
6,test,q2_mid_low,96,0.2297
7,test,q3_mid_high,99,0.2368
8,test,q4_high,108,0.2584
9,test,missing,1,0.0024


### Survival by FareLog bins within Pclass

- Hypothesis / theory check: train-side survival rates by `FareLogBinTrain` should be read within `Pclass`, not as an unconditional feature decision.
- Data used: `train.csv` only.


In [14]:
farelog_pclass_survival = (
    fare_train.groupby(['Pclass', 'FareLogBinTrain'], dropna=False)['Survived']
    .agg(count='count', survived_sum='sum', survival_rate='mean')
    .reset_index()
    .sort_values(['Pclass', 'FareLogBinTrain'])
)
display(farelog_pclass_survival.round(4))


,Pclass,FareLogBinTrain,count,survived_sum,survival_rate
0,1,q1_low,6,0,0.0000
1,1,q3_mid_high,51,27,0.5294
2,1,q4_high,159,109,0.6855
3,2,q1_low,6,0,0.0000
4,2,q2_mid_low,86,33,0.3837
5,2,q3_mid_high,70,42,0.6000
6,2,q4_high,22,12,0.5455
7,3,q1_low,211,44,0.2085
8,3,q2_mid_low,138,35,0.2536
9,3,q3_mid_high,101,32,0.3168


### Survival by FareLog bins within Sex x Pclass

- Hypothesis / theory check: if fare signal is mostly socioeconomic duplication, patterns should weaken or become uneven inside `Sex x Pclass` slices.
- Data used: `train.csv` only; the table is bounded to the observed `Sex x Pclass x FareLogBinTrain` groups.


In [15]:
farelog_sex_pclass_survival = (
    fare_train.groupby(['Sex', 'Pclass', 'FareLogBinTrain'], dropna=False)['Survived']
    .agg(count='count', survived_sum='sum', survival_rate='mean')
    .reset_index()
    .sort_values(['Sex', 'Pclass', 'FareLogBinTrain'])
)
display(farelog_sex_pclass_survival.round(4))


,Sex,Pclass,FareLogBinTrain,count,survived_sum,survival_rate
0,female,1,q3_mid_high,8,7,0.8750
1,female,1,q4_high,86,84,0.9767
2,female,2,q2_mid_low,29,26,0.8966
3,female,2,q3_mid_high,38,35,0.9211
4,female,2,q4_high,9,9,1.0000
5,female,3,q1_low,43,30,0.6977
6,female,3,q2_mid_low,38,17,0.4474
7,female,3,q3_mid_high,47,23,0.4894
8,female,3,q4_high,16,2,0.1250
9,male,1,q1_low,6,0,0.0000


### Current Fare/FareLog reading after EDA and frozen context

- Fare has visible target-side signal in EDA.
- However, the current raw_tabular leader already includes raw Fare.
- Therefore FareLog is not a new information source; it is only a transformation of an already-used feature.
- In the old repo, FareLog gave gains mainly in RF / gated-RF style paths.
- In the clean repo, GradientBoostingClassifier has been the leader from the beginning and may already capture much of the small nonlinear Fare signal from raw Fare.
- FareLog should therefore be treated as a low-expectation controlled check candidate, not as a strong EDA discovery.
- Any decision on FareLog must be made by a controlled train-side comparison first, and only then by frozen submission if it survives the protocol.
- Do not make a feature acceptance/rejection decision inside EDA.


### Fare overlap with Embarked, Sex, CabinKnown, FamilySize

- Hypothesis / theory check: fare structure may be partly explained by embarkation, sex, cabin availability, and family size.
- Data used: `train.csv` and `test.csv`; grouped structural diagnostics only.
- Boundary: `FamilySize` is diagnostic only inside this Fare/Cabin EDA block and is not a feature promotion.


In [16]:
def fare_overlap_summary(frame, group_col, dataset_name):
    summary = (
        frame.groupby(group_col, dropna=False)
        .agg(
            count=('PassengerId', 'count'),
            fare_missing=('Fare', lambda s: s.isna().sum()),
            fare_median=('Fare', 'median'),
            fare_q25=('Fare', lambda s: s.quantile(0.25)),
            fare_q75=('Fare', lambda s: s.quantile(0.75)),
            farelog_median=('FareLog', 'median'),
            pclass_median=('Pclass', 'median'),
            cabin_known_rate=('CabinKnown', 'mean'),
        )
        .reset_index()
        .rename(columns={group_col: 'group_value'})
    )
    summary.insert(0, 'feature', group_col)
    summary.insert(0, 'dataset', dataset_name)
    return summary

for group_col in ['Embarked', 'Sex', 'CabinKnown', 'FamilySize']:
    display(
        pd.concat(
            [fare_overlap_summary(frame, group_col, dataset_name) for dataset_name, frame in fare_datasets],
            ignore_index=True,
        ).round(4)
    )


,dataset,feature,group_value,count,fare_missing,fare_median,fare_q25,fare_q75,farelog_median,pclass_median,cabin_known_rate
0,train,Embarked,C,168,0,29.7000,13.6980,78.5000,3.4243,1.0,0.4107
1,train,Embarked,Q,77,0,7.7500,7.7500,15.5000,2.1691,3.0,0.0519
2,train,Embarked,S,644,0,13.0000,8.0500,27.9000,2.6391,3.0,0.2003
3,train,Embarked,NaN,2,0,80.0000,80.0000,80.0000,4.3944,1.0,1.0000
4,test,Embarked,C,102,0,27.7208,13.8594,78.4729,3.3576,1.0,0.4804
5,test,Embarked,Q,46,0,7.7500,7.7500,7.8792,2.1691,3.0,0.0217
6,test,Embarked,S,270,1,13.7750,8.0500,26.5500,2.6929,3.0,0.1519


,dataset,feature,group_value,count,fare_missing,fare_median,fare_q25,fare_q75,farelog_median,pclass_median,cabin_known_rate
0,train,Sex,female,314,0,23.0000,12.0719,55.0000,3.1781,2.0,0.3089
1,train,Sex,male,577,0,10.5000,7.8958,26.5500,2.4423,3.0,0.1854
2,test,Sex,female,152,0,21.5125,8.6260,55.4417,3.1138,2.0,0.2895
3,test,Sex,male,266,1,13.0000,7.8542,26.5500,2.6391,3.0,0.1767


,dataset,feature,group_value,count,fare_missing,fare_median,fare_q25,fare_q75,farelog_median,pclass_median,cabin_known_rate
0,train,CabinKnown,0,687,0,10.5000,7.8771,23.0000,2.4423,3.0,0.0
1,train,CabinKnown,1,204,0,55.2208,29.4531,89.3282,4.0293,1.0,1.0
2,test,CabinKnown,0,327,1,11.5000,7.8229,22.2750,2.5257,3.0,0.0
3,test,CabinKnown,1,91,0,61.9792,30.1000,134.5000,4.1428,1.0,1.0


,dataset,feature,group_value,count,fare_missing,fare_median,fare_q25,fare_q75,farelog_median,pclass_median,cabin_known_rate
0,train,FamilySize,1,537,0,8.1375,7.7750,15.0000,2.2124,3.0,0.1750
1,train,FamilySize,2,161,0,26.0000,15.5500,63.3583,3.2958,2.0,0.4099
2,train,FamilySize,3,102,0,24.1500,15.7417,46.8750,3.2249,2.0,0.2843
3,train,FamilySize,4,29,0,27.7500,20.5750,65.0000,3.3586,2.0,0.3103
4,train,FamilySize,5,15,0,25.4667,22.0375,34.3750,3.2759,3.0,0.1333
5,train,FamilySize,6,22,0,29.1250,27.9000,39.6875,3.4054,3.0,0.1818
6,train,FamilySize,7,12,0,31.2750,31.2750,31.3875,3.4743,3.0,0.0000
7,train,FamilySize,8,6,0,46.9000,46.9000,46.9000,3.8691,3.0,0.0000
8,train,FamilySize,11,7,0,69.5500,69.5500,69.5500,4.2563,3.0,0.0000
9,test,FamilySize,1,253,1,8.0500,7.7500,17.2500,2.2028,3.0,0.1462


### CabinKnown structural and target-side overlap

- Hypothesis / theory check: `CabinKnown` may reflect ticketing/class structure and should be audited against `Pclass`, `FareLogBinTrain`, `Sex`, and diagnostic `FamilySize`.
- Data used: train/test for structural counts; train only for survival rates.


In [17]:
def distribution_table(frame, column, dataset_name, order=None):
    counts = frame[column].value_counts(dropna=False)
    if order is not None:
        counts = counts.reindex(order, fill_value=0)
    result = counts.rename_axis(column).reset_index(name='count')
    result.insert(0, 'dataset', dataset_name)
    result['pct'] = result['count'] / len(frame)
    return result

def cross_count_pct(frame, group_cols, dataset_name):
    result = (
        frame.groupby(group_cols, dropna=False)
        .agg(count=('PassengerId', 'count'))
        .reset_index()
        .sort_values(group_cols)
    )
    result.insert(0, 'dataset', dataset_name)
    result['pct'] = result['count'] / len(frame)
    return result

cabin_dist = pd.concat(
    [distribution_table(frame, 'CabinKnown', dataset_name, order=[0, 1]) for dataset_name, frame in fare_datasets],
    ignore_index=True,
)
display(cabin_dist.round(4))

cabin_rate_by_pclass = []
for dataset_name, frame in fare_datasets:
    summary = (
        frame.groupby('Pclass', dropna=False)
        .agg(count=('PassengerId', 'count'), cabin_known_rate=('CabinKnown', 'mean'))
        .reset_index()
    )
    summary.insert(0, 'dataset', dataset_name)
    cabin_rate_by_pclass.append(summary)
display(pd.concat(cabin_rate_by_pclass, ignore_index=True).round(4))

display(
    fare_train.groupby(['Pclass', 'CabinKnown'], dropna=False)['Survived']
    .agg(count='count', survived_sum='sum', survival_rate='mean')
    .reset_index()
    .sort_values(['Pclass', 'CabinKnown'])
    .round(4)
)

display(pd.concat([cross_count_pct(frame, ['CabinKnown', 'FareLogBinTrain'], dataset_name) for dataset_name, frame in fare_datasets], ignore_index=True).round(4))
display(pd.concat([cross_count_pct(frame, ['CabinKnown', 'Sex'], dataset_name) for dataset_name, frame in fare_datasets], ignore_index=True).round(4))
display(pd.concat([cross_count_pct(frame, ['CabinKnown', 'FamilySize'], dataset_name) for dataset_name, frame in fare_datasets], ignore_index=True).round(4))


,dataset,CabinKnown,count,pct
0,train,0,687,0.7710
1,train,1,204,0.2290
2,test,0,327,0.7823
3,test,1,91,0.2177


,dataset,Pclass,count,cabin_known_rate
0,train,1,216,0.8148
1,train,2,184,0.0870
2,train,3,491,0.0244
3,test,1,107,0.7477
4,test,2,93,0.0753
5,test,3,218,0.0183


,Pclass,CabinKnown,count,survived_sum,survival_rate
0,1,0,40,19,0.4750
1,1,1,176,117,0.6648
2,2,0,168,74,0.4405
3,2,1,16,13,0.8125
4,3,0,479,113,0.2359
5,3,1,12,6,0.5000


,dataset,CabinKnown,FareLogBinTrain,count,pct
0,train,0,q1_low,215,0.2413
1,train,0,q2_mid_low,208,0.2334
2,train,0,q3_mid_high,181,0.2031
3,train,0,q4_high,83,0.0932
4,train,1,q1_low,8,0.0090
5,train,1,q2_mid_low,16,0.0180
6,train,1,q3_mid_high,41,0.0460
7,train,1,q4_high,139,0.1560
8,test,0,missing,1,0.0024
9,test,0,q1_low,110,0.2632


,dataset,CabinKnown,Sex,count,pct
0,train,0,female,217,0.2435
1,train,0,male,470,0.5275
2,train,1,female,97,0.1089
3,train,1,male,107,0.1201
4,test,0,female,108,0.2584
5,test,0,male,219,0.5239
6,test,1,female,44,0.1053
7,test,1,male,47,0.1124


,dataset,CabinKnown,FamilySize,count,pct
0,train,0,1,443,0.4972
1,train,0,2,95,0.1066
2,train,0,3,73,0.0819
3,train,0,4,20,0.0224
4,train,0,5,13,0.0146
5,train,0,6,18,0.0202
6,train,0,7,12,0.0135
7,train,0,8,6,0.0067
8,train,0,11,7,0.0079
9,train,1,1,94,0.1055


### Deck diagnostic only

- Hypothesis / theory check: parsed cabin deck may expose structure, but sparse deck levels are fragile.
- Data used: train/test structural counts; target-side train table is shown only for deck counts large enough to read.
- Boundary: `Deck` is diagnostic only inside this block and is not promoted as a feature.


In [18]:
def add_deck(frame):
    out = frame.copy()
    out['Deck'] = out['Cabin'].map(lambda value: 'Unknown' if pd.isna(value) else str(value).strip()[0])
    return out

fare_train = add_deck(fare_train)
fare_test = add_deck(fare_test)
fare_datasets = [('train', fare_train), ('test', fare_test)]

deck_order = sorted(set(fare_train['Deck']).union(set(fare_test['Deck'])))
deck_distribution = pd.concat(
    [distribution_table(frame, 'Deck', dataset_name, order=deck_order) for dataset_name, frame in fare_datasets],
    ignore_index=True,
)
display(deck_distribution.round(4))

min_deck_count = 10
deck_survival = (
    fare_train.groupby('Deck', dropna=False)['Survived']
    .agg(count='count', survived_sum='sum', survival_rate='mean')
    .reset_index()
    .sort_values('Deck')
)
print('Deck survival rates shown only for train Deck counts >= %s.' % min_deck_count)
display(deck_survival[deck_survival['count'] >= min_deck_count].round(4))
display(deck_survival[deck_survival['count'] < min_deck_count][['Deck', 'count']].rename(columns={'count': 'count_below_target_rate_threshold'}))


,dataset,Deck,count,pct
0,train,A,15,0.0168
1,train,B,47,0.0527
2,train,C,59,0.0662
3,train,D,33,0.0370
4,train,E,32,0.0359
5,train,F,13,0.0146
6,train,G,4,0.0045
7,train,T,1,0.0011
8,train,Unknown,687,0.7710
9,test,A,7,0.0167


Deck survival rates shown only for train Deck counts >= 10.


,Deck,count,survived_sum,survival_rate
0,A,15,7,0.4667
1,B,47,35,0.7447
2,C,59,35,0.5932
3,D,33,25,0.7576
4,E,32,24,0.7500
5,F,13,8,0.6154
8,Unknown,687,206,0.2999


,Deck,count_below_target_rate_threshold
6,G,4
7,T,1


### Joint overlap: Pclass + FareLogBin + CabinKnown

- Hypothesis / theory check: the joint table is the main audit for whether Fare/Cabin structure adds marginal signal beyond broad socioeconomic status or mainly duplicates it.
- Data used: train only for survival rate; train/test for structural presence and alignment.


In [19]:
joint_cols = ['Pclass', 'FareLogBinTrain', 'CabinKnown']

joint_survival = (
    fare_train.groupby(joint_cols, dropna=False)['Survived']
    .agg(count='count', survived_sum='sum', survival_rate='mean')
    .reset_index()
    .sort_values(joint_cols)
)
display(joint_survival.round(4))

joint_alignment = pd.concat(
    [cross_count_pct(frame, joint_cols, dataset_name) for dataset_name, frame in fare_datasets],
    ignore_index=True,
)
joint_alignment_wide = joint_alignment.pivot_table(
    index=joint_cols,
    columns='dataset',
    values=['count', 'pct'],
    fill_value=0,
).reset_index()
flat_joint_columns = []
for col in joint_alignment_wide.columns:
    if isinstance(col, tuple):
        flat_joint_columns.append(col[0] if col[1] == '' else '%s_%s' % (col[0], col[1]))
    else:
        flat_joint_columns.append(col)
joint_alignment_wide.columns = flat_joint_columns
display(joint_alignment_wide.sort_values(joint_cols).round(4))


,Pclass,FareLogBinTrain,CabinKnown,count,survived_sum,survival_rate
0,1,q1_low,0,2,0,0.0000
1,1,q1_low,1,4,0,0.0000
2,1,q3_mid_high,0,16,6,0.3750
3,1,q3_mid_high,1,35,21,0.6000
4,1,q4_high,0,22,13,0.5909
5,1,q4_high,1,137,96,0.7007
6,2,q1_low,0,6,0,0.0000
7,2,q2_mid_low,0,75,24,0.3200
8,2,q2_mid_low,1,11,9,0.8182
9,2,q3_mid_high,0,67,40,0.5970


,Pclass,FareLogBinTrain,CabinKnown,count_test,count_train,pct_test,pct_train
0,1,q1_low,0,1.0,2.0,0.0024,0.0022
1,1,q1_low,1,1.0,4.0,0.0024,0.0045
2,1,q3_mid_high,0,12.0,16.0,0.0287,0.0180
3,1,q3_mid_high,1,14.0,35.0,0.0335,0.0393
4,1,q4_high,0,14.0,22.0,0.0335,0.0247
5,1,q4_high,1,65.0,137.0,0.1555,0.1538
6,2,q1_low,0,0.0,6.0,0.0000,0.0067
7,2,q2_mid_low,0,38.0,75.0,0.0909,0.0842
8,2,q2_mid_low,1,4.0,11.0,0.0096,0.0123
9,2,q3_mid_high,0,33.0,67.0,0.0789,0.0752


### CabinKnown subgroup gate rationale: male Pclass 1 CabinKnown=0 downshift

Full `CabinKnown` is a broad socioeconomic signal and can be too aggressive. EDA and Step 16 showed that `CabinKnown` has signal, but it overlaps strongly with `Pclass` and `Fare`; after full-feature public-transfer failure, only bounded subgroup corrections should be considered.

The selected frozen subgroup candidate is:

```python
raw_tabular_pred == 1
and raw_plus_cabinknown_pred == 0
and Sex == "male"
and Pclass == 1
and CabinKnown == 0
```

Interpretation:

- This is not a raw PassengerId correction.
- This is a model-diff subgroup: the GB baseline predicts survival, while the CabinKnown-aware GB downshifts to death.
- The subgroup is consistent with the EDA concern that missing `Cabin` in `Pclass == 1` can separate weaker first-class socioeconomic/cabin signal from high-confidence first-class survivors.
- The subgroup is small and bounded.

Known OOF/group-diagnostic evidence before frozen public scoring:

- OOF changed rows: 6
- rescue/kill/net: 5 / 1 / +4
- train PassengerIds: 31 35 156 296 448 794
- matching test changed PassengerIds: 915 1040 1215

Reading boundary:

- This EDA note does not introduce Deck/Ticket/Family.
- This EDA note does not justify more subgroup search.
- This subgroup is a single frozen gate candidate.
- Public score 0.79904 is recorded only after the candidate was already identified by OOF/group diagnostics.


### Reading boundary

- This block is EDA only.
- No feature is accepted/rejected here.
- FareLog is only a transformation candidate because raw Fare is already in raw_tabular.
- The next fair step for FareLog, if pursued, is controlled train-side comparison against raw_tabular.
- If it survives train-side protocol, a small frozen public check may be used to close the branch honestly.
- No public-score tuning or micro-variants after failed transfer.


## TicketPrefix Risk

- Hypothesis / theory check: `TicketPrefix` may be high-cardinality and unstable across train/test.
- Data used: structural counts from `train.csv` and `test.csv` without test `Survived`.
- Risk / limitation: rare and unseen prefixes can overfit or break alignment.
- Boundary: this is not a final model decision.

### TicketPrefix risk semantics

`TicketPrefix` is parsed to preserve ticket-family structure while removing punctuation noise.

Rules:

- purely numeric tickets are assigned to `NONE`, because they have no prefix;
- punctuation is removed inside prefix tokens, so `A/5` becomes `A5` and `C.A.` becomes `CA`;
- multi-part non-numeric prefixes are preserved instead of collapsing everything to the first letter.

The goal is to test whether ticket-family structure exists and whether it is too sparse or unstable between train and test. This block is a risk check, not a promotion of `TicketPrefix` as a final model feature.

In [20]:
train_ticket = train["TicketPrefix"].value_counts(dropna=False).rename("train_count")
test_ticket = test["TicketPrefix"].value_counts(dropna=False).rename("test_count")
ticket = pd.concat([train_ticket, test_ticket], axis=1).fillna(0).astype(int).reset_index(names="TicketPrefix")
ticket["train_share"] = ticket["train_count"] / len(train)
ticket["test_share"] = ticket["test_count"] / len(test)
ticket["rare"] = (ticket["train_count"].between(1, 9)) | (ticket["test_count"].between(1, 4))
ticket["unseen_in_train"] = ticket["train_count"].eq(0) & ticket["test_count"].gt(0)
ticket["unseen_in_test"] = ticket["train_count"].gt(0) & ticket["test_count"].eq(0)

ticket_risk_summary = pd.DataFrame(
    [
        {
            "categories": len(ticket),
            "rare_categories": int(ticket["rare"].sum()),
            "unseen_in_train": int(ticket["unseen_in_train"].sum()),
            "unseen_in_test": int(ticket["unseen_in_test"].sum()),
        }
    ]
)

display(ticket.sort_values(["rare", "train_count"], ascending=[False, False]).head(40))
display(ticket_risk_summary)

,TicketPrefix,train_count,test_count,train_share,test_share,rare,unseen_in_train,unseen_in_test
5,STONO,12,2,0.013468,0.004785,True,False,False
8,A4,7,3,0.007856,0.007177,True,False,False
9,STONO2,6,1,0.006734,0.002392,True,False,False
10,SOC,6,2,0.006734,0.004785,True,False,False
11,C,5,3,0.005612,0.007177,True,False,False
12,FCC,5,4,0.005612,0.009569,True,False,False
13,LINE,4,0,0.004489,0.000000,True,False,True
14,PP,3,1,0.003367,0.002392,True,False,False
15,WEP,3,1,0.003367,0.002392,True,False,False
16,SOPP,3,4,0.003367,0.009569,True,False,False


,categories,rare_categories,unseen_in_train,unseen_in_test
0,37,30,6,10


Interpretation: `TicketPrefix` preserves some structural grouping, but high rare/unseen counts indicate instability risk. It should not be used directly as a categorical model feature without additional validation or gating.

## Step 17A Family/Surname Structural EDA

- Hypothesis / theory check: `FamilySize` and full-manifest `SurnameCount` are overlapping but not identical structural signals.
- Data used: `train.csv` and `test.csv` for structural distributions; `Survived` only in train-side survival summaries.
- Allowed boundary: `FamilySize`, `FamilyBand`, parsed `Surname`, full-manifest `SurnameCount`, `SurnameCountBand`, combined `FamilySurnameBand`, and equality/mismatch diagnostics.
- Excluded boundary: no Fare/FareLog, Ticket, target encoding, surname survival features, PassengerId correction rules, models, submissions, or public-score tuning.
- Band convention: reuse the project `bucket_family_size` cut points for both `FamilyBand` and `SurnameCountBand`: `alone=1`, `small=2-4`, `medium=5-6`, `large=7+`.


In [21]:
def add_family_surname_structure(train_frame, test_frame):
    manifest = pd.concat(
        [
            train_frame.assign(Dataset="train"),
            test_frame.assign(Dataset="test"),
        ],
        ignore_index=True,
        sort=False,
    )
    manifest["FamilySize"] = manifest["SibSp"] + manifest["Parch"] + 1
    manifest["FamilyBand"] = manifest["FamilySize"].map(bucket_family_size)
    manifest["Surname"] = manifest["Name"].str.extract(r"^([^,]+),", expand=False).str.strip()

    surname_counts = manifest["Surname"].value_counts(dropna=False)
    manifest["SurnameCount"] = manifest["Surname"].map(surname_counts).astype(int)
    manifest["SurnameCountBand"] = manifest["SurnameCount"].map(bucket_family_size)
    manifest["FamilySizeEqualsSurnameCount"] = manifest["FamilySize"].eq(manifest["SurnameCount"])
    manifest["FamilySizeDiffersSurnameCount"] = ~manifest["FamilySizeEqualsSurnameCount"]
    manifest["FamilySurnameBand"] = np.where(
        manifest["FamilySizeEqualsSurnameCount"],
        "match_" + manifest["FamilyBand"].astype(str),
        "mismatch_family_" + manifest["FamilyBand"].astype(str) + "_surname_" + manifest["SurnameCountBand"].astype(str),
    )
    return (
        manifest.loc[manifest["Dataset"].eq("train")].copy(),
        manifest.loc[manifest["Dataset"].eq("test")].copy(),
        manifest,
    )


family_train, family_test, family_manifest = add_family_surname_structure(train_raw, test_raw)


def structural_distribution(frame, column, dataset_name):
    out = frame[column].value_counts(dropna=False).rename_axis(column).reset_index(name="count")
    out["rate"] = out["count"] / len(frame)
    if pd.api.types.is_numeric_dtype(out[column]):
        out = out.sort_values(column)
    else:
        out = out.sort_values(["count", column], ascending=[False, True])
    out.insert(0, "dataset", dataset_name)
    return out.reset_index(drop=True)


def structural_rate_table(frame, group_cols):
    return (
        frame.groupby(group_cols, dropna=False)["Survived"]
        .agg(count="count", survived_count="sum", survival_rate="mean")
        .reset_index()
        .sort_values(group_cols)
    )


def structural_segment_distribution(frame, group_cols, dataset_name):
    out = (
        frame.groupby(group_cols + ["FamilySurnameBand"], dropna=False)
        .agg(count=("PassengerId", "count"))
        .reset_index()
    )
    out["rate"] = out["count"] / out.groupby(group_cols)["count"].transform("sum")
    out.insert(0, "dataset", dataset_name)
    return out.sort_values(group_cols + ["FamilySurnameBand"]).reset_index(drop=True)


def structural_bucket_presence(train_frame, test_frame, column):
    out = pd.DataFrame(
        {
            "train_count": train_frame[column].value_counts(dropna=False),
            "test_count": test_frame[column].value_counts(dropna=False),
        }
    ).fillna(0).astype(int)
    out["train_rate"] = out["train_count"] / len(train_frame)
    out["test_rate"] = out["test_count"] / len(test_frame)
    return out.rename_axis(column).reset_index().sort_values(column).reset_index(drop=True)


display(pd.DataFrame({"dataset": ["train", "test", "full_manifest"], "rows": [len(family_train), len(family_test), len(family_manifest)]}))


,dataset,rows
0,train,891
1,test,418
2,full_manifest,1309


### Feature Sanity

Structural feature construction only. `SurnameCount` is counted on the combined train/test manifest and does not use `Survived`.


In [22]:
for column in ["FamilySize", "FamilyBand", "SurnameCount", "SurnameCountBand"]:
    display(pd.concat(
        [
            structural_distribution(family_train, column, "train"),
            structural_distribution(family_test, column, "test"),
        ],
        ignore_index=True,
    ).round(4))

surname_example_cols = ["Dataset", "PassengerId", "Name", "Surname", "FamilySize", "SurnameCount"]
display(family_manifest[surname_example_cols].head(12))


,dataset,FamilySize,count,rate
0,train,1,537,0.6027
1,train,2,161,0.1807
2,train,3,102,0.1145
3,train,4,29,0.0325
4,train,5,15,0.0168
5,train,6,22,0.0247
6,train,7,12,0.0135
7,train,8,6,0.0067
8,train,11,7,0.0079
9,test,1,253,0.6053


,dataset,FamilyBand,count,rate
0,train,alone,537,0.6027
1,train,small,292,0.3277
2,train,medium,37,0.0415
3,train,large,25,0.0281
4,test,alone,253,0.6053
5,test,small,145,0.3469
6,test,large,10,0.0239
7,test,medium,10,0.0239


,dataset,SurnameCount,count,rate
0,train,1,447,0.5017
1,train,2,168,0.1886
2,train,3,121,0.1358
3,train,4,62,0.0696
4,train,5,19,0.0213
5,train,6,45,0.0505
6,train,7,3,0.0034
7,train,8,10,0.0112
8,train,11,16,0.0180
9,test,1,190,0.4545


,dataset,SurnameCountBand,count,rate
0,train,alone,447,0.5017
1,train,small,351,0.3939
2,train,medium,64,0.0718
3,train,large,29,0.0325
4,test,small,192,0.4593
5,test,alone,190,0.4545
6,test,medium,20,0.0478
7,test,large,16,0.0383


,Dataset,PassengerId,Name,Surname,FamilySize,SurnameCount
0,train,1,"Braund, Mr. Owen Harris",Braund,2,2
1,train,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Cumings,2,2
2,train,3,"Heikkinen, Miss. Laina",Heikkinen,1,1
3,train,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Futrelle,2,2
4,train,5,"Allen, Mr. William Henry",Allen,1,2
5,train,6,"Moran, Mr. James",Moran,1,3
6,train,7,"McCarthy, Mr. Timothy J",McCarthy,1,2
7,train,8,"Palsson, Master. Gosta Leonard",Palsson,5,5
8,train,9,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",Johnson,3,6
9,train,10,"Nasser, Mrs. Nicholas (Adele Achem)",Nasser,2,2


### Direct Overlap Checks

These tables quantify direct redundancy and divergence between passenger-declared family size and same-surname structure.


In [23]:
display(pd.crosstab(family_train["FamilySize"], family_train["SurnameCount"], margins=True))
display(pd.crosstab(family_test["FamilySize"], family_test["SurnameCount"], margins=True))
display(pd.crosstab(family_train["FamilyBand"], family_train["SurnameCountBand"], margins=True))
display(pd.crosstab(family_test["FamilyBand"], family_test["SurnameCountBand"], margins=True))

for column in ["FamilySizeEqualsSurnameCount", "FamilySizeDiffersSurnameCount"]:
    display(pd.concat(
        [
            structural_distribution(family_train, column, "train"),
            structural_distribution(family_test, column, "test"),
        ],
        ignore_index=True,
    ).round(4))

mismatch_cols = [
    "Dataset",
    "PassengerId",
    "Name",
    "Sex",
    "Pclass",
    "SibSp",
    "Parch",
    "FamilySize",
    "Surname",
    "SurnameCount",
    "FamilyBand",
    "SurnameCountBand",
    "FamilySurnameBand",
]
display(pd.concat(
    [
        family_train.loc[family_train["FamilySizeDiffersSurnameCount"], mismatch_cols].head(10),
        family_test.loc[family_test["FamilySizeDiffersSurnameCount"], mismatch_cols].head(10),
    ],
    ignore_index=True,
))


SurnameCount,1,2,3,4,5,6,7,8,11,All
FamilySize,,,,,,,,,,
1,426,49,29,15,7,9,1,0,1,537
2,16,107,25,9,2,2,0,0,0,161
3,5,9,66,15,0,5,2,0,0,102
4,0,3,0,22,0,4,0,0,0,29
5,0,0,0,1,10,4,0,0,0,15
6,0,0,1,0,0,21,0,0,0,22
7,0,0,0,0,0,0,0,4,8,12
8,0,0,0,0,0,0,0,6,0,6
11,0,0,0,0,0,0,0,0,7,7


SurnameCount,1,2,3,4,5,6,7,8,11,All
FamilySize,,,,,,,,,,
1,184,38,17,7,3,1,1,1,1,253
2,4,52,10,4,2,2,0,0,0,74
3,2,7,40,2,1,2,3,0,0,57
4,0,1,0,13,0,0,0,0,0,14
5,0,0,1,0,5,1,0,0,0,7
6,0,0,0,0,0,3,0,0,0,3
7,0,0,0,0,0,0,0,3,1,4
8,0,0,0,0,0,0,0,2,0,2
11,0,0,0,0,0,0,0,0,4,4


SurnameCountBand,alone,large,medium,small,All
FamilyBand,,,,,
alone,426,2,16,93,537
large,0,25,0,0,25
medium,0,0,35,2,37
small,21,2,13,256,292
All,447,29,64,351,891


SurnameCountBand,alone,large,medium,small,All
FamilyBand,,,,,
alone,184,3,4,62,253
large,0,10,0,0,10
medium,0,0,9,1,10
small,6,3,7,129,145
All,190,16,20,192,418


,dataset,FamilySizeEqualsSurnameCount,count,rate
0,train,False,226,0.2536
1,train,True,665,0.7464
2,test,False,115,0.2751
3,test,True,303,0.7249


,dataset,FamilySizeDiffersSurnameCount,count,rate
0,train,False,665,0.7464
1,train,True,226,0.2536
2,test,False,303,0.7249
3,test,True,115,0.2751


,Dataset,PassengerId,Name,Sex,Pclass,SibSp,Parch,FamilySize,Surname,SurnameCount,FamilyBand,SurnameCountBand,FamilySurnameBand
0,train,5,"Allen, Mr. William Henry",male,3,0,0,1,Allen,2,alone,small,mismatch_family_alone_surname_small
1,train,6,"Moran, Mr. James",male,3,0,0,1,Moran,3,alone,small,mismatch_family_alone_surname_small
2,train,7,"McCarthy, Mr. Timothy J",male,1,0,0,1,McCarthy,2,alone,small,mismatch_family_alone_surname_small
3,train,9,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,3,0,2,3,Johnson,6,small,medium,mismatch_family_small_surname_medium
4,train,12,"Bonnell, Miss. Elizabeth",female,1,0,0,1,Bonnell,2,alone,small,mismatch_family_alone_surname_small
5,train,14,"Andersson, Mr. Anders Johan",male,3,1,5,7,Andersson,11,large,large,mismatch_family_large_surname_large
6,train,18,"Williams, Mr. Charles Eugene",male,2,0,0,1,Williams,5,alone,medium,mismatch_family_alone_surname_medium
7,train,19,"Vander Planke, Mrs. Julius (Emelia Maria Vande...",female,3,1,0,2,Vander Planke,4,small,small,mismatch_family_small_surname_small
8,train,23,"McGowan, Miss. Anna ""Annie""",female,3,0,0,1,McGowan,2,alone,small,mismatch_family_alone_surname_small
9,train,26,"Asplund, Mrs. Carl Oscar (Selma Augusta Emilia...",female,3,1,5,7,Asplund,8,large,large,mismatch_family_large_surname_large


### Train/Test Distribution Alignment

Distribution checks for individual structural fields and the proposed combined `FamilySurnameBand` bucket.


In [24]:
for column in [
    "FamilySize",
    "SurnameCount",
    "FamilyBand",
    "SurnameCountBand",
    "FamilySizeEqualsSurnameCount",
    "FamilySizeDiffersSurnameCount",
    "FamilySurnameBand",
]:
    display(structural_bucket_presence(family_train, family_test, column).round(4))


,FamilySize,train_count,test_count,train_rate,test_rate
0,1,537,253,0.6027,0.6053
1,2,161,74,0.1807,0.1770
2,3,102,57,0.1145,0.1364
3,4,29,14,0.0325,0.0335
4,5,15,7,0.0168,0.0167
5,6,22,3,0.0247,0.0072
6,7,12,4,0.0135,0.0096
7,8,6,2,0.0067,0.0048
8,11,7,4,0.0079,0.0096


,SurnameCount,train_count,test_count,train_rate,test_rate
0,1,447,190,0.5017,0.4545
1,2,168,98,0.1886,0.2344
2,3,121,68,0.1358,0.1627
3,4,62,26,0.0696,0.0622
4,5,19,11,0.0213,0.0263
5,6,45,9,0.0505,0.0215
6,7,3,4,0.0034,0.0096
7,8,10,6,0.0112,0.0144
8,11,16,6,0.0180,0.0144


,FamilyBand,train_count,test_count,train_rate,test_rate
0,alone,537,253,0.6027,0.6053
1,large,25,10,0.0281,0.0239
2,medium,37,10,0.0415,0.0239
3,small,292,145,0.3277,0.3469


,SurnameCountBand,train_count,test_count,train_rate,test_rate
0,alone,447,190,0.5017,0.4545
1,large,29,16,0.0325,0.0383
2,medium,64,20,0.0718,0.0478
3,small,351,192,0.3939,0.4593


,FamilySizeEqualsSurnameCount,train_count,test_count,train_rate,test_rate
0,False,226,115,0.2536,0.2751
1,True,665,303,0.7464,0.7249


,FamilySizeDiffersSurnameCount,train_count,test_count,train_rate,test_rate
0,False,665,303,0.7464,0.7249
1,True,226,115,0.2536,0.2751


,FamilySurnameBand,train_count,test_count,train_rate,test_rate
0,match_alone,426,184,0.4781,0.4402
1,match_large,13,6,0.0146,0.0144
2,match_medium,31,8,0.0348,0.0191
3,match_small,195,105,0.2189,0.2512
4,mismatch_family_alone_surname_large,2,3,0.0022,0.0072
5,mismatch_family_alone_surname_medium,16,4,0.0180,0.0096
6,mismatch_family_alone_surname_small,93,62,0.1044,0.1483
7,mismatch_family_large_surname_large,12,4,0.0135,0.0096
8,mismatch_family_medium_surname_medium,4,1,0.0045,0.0024
9,mismatch_family_medium_surname_small,2,1,0.0022,0.0024


### Train-Only Survival Analysis

`Survived` is used only for train-side EDA reporting. No survival-derived feature values are created.


In [25]:
for group_cols in [
    ["FamilyBand"],
    ["SurnameCountBand"],
    ["FamilySizeEqualsSurnameCount"],
    ["FamilySizeDiffersSurnameCount"],
    ["FamilyBand", "SurnameCountBand"],
    ["FamilySurnameBand"],
]:
    display(structural_rate_table(family_train, group_cols).round(4))


,FamilyBand,count,survived_count,survival_rate
0,alone,537,163.0,0.3035
1,large,25,4.0,0.1600
2,medium,37,6.0,0.1622
3,small,292,169.0,0.5788


,SurnameCountBand,count,survived_count,survival_rate
0,alone,447,149.0,0.3333
1,large,29,6.0,0.2069
2,medium,64,20.0,0.3125
3,small,351,167.0,0.4758


,FamilySizeEqualsSurnameCount,count,survived_count,survival_rate
0,False,226,94.0,0.4159
1,True,665,248.0,0.3729


,FamilySizeDiffersSurnameCount,count,survived_count,survival_rate
0,False,665,248.0,0.3729
1,True,226,94.0,0.4159


,FamilyBand,SurnameCountBand,count,survived_count,survival_rate
0,alone,alone,426,132.0,0.3099
1,alone,large,2,1.0,0.5000
2,alone,medium,16,7.0,0.4375
3,alone,small,93,23.0,0.2473
4,large,large,25,4.0,0.1600
5,medium,medium,35,4.0,0.1143
6,medium,small,2,2.0,1.0000
7,small,alone,21,17.0,0.8095
8,small,large,2,1.0,0.5000
9,small,medium,13,9.0,0.6923


,FamilySurnameBand,count,survived_count,survival_rate
0,match_alone,426,132.0,0.3099
1,match_large,13,0.0,0.0000
2,match_medium,31,4.0,0.1290
3,match_small,195,112.0,0.5744
4,mismatch_family_alone_surname_large,2,1.0,0.5000
5,mismatch_family_alone_surname_medium,16,7.0,0.4375
6,mismatch_family_alone_surname_small,93,23.0,0.2473
7,mismatch_family_large_surname_large,12,4.0,0.3333
8,mismatch_family_medium_surname_medium,4,0.0,0.0000
9,mismatch_family_medium_surname_small,2,2.0,1.0000


### Segment Checks

Combined structural buckets by `Sex`, `Pclass`, and `Sex x Pclass`, with train-side survival focus on female Pclass 3 and male Pclass 1.


In [26]:
for group_cols in [["Sex"], ["Pclass"], ["Sex", "Pclass"]]:
    display(pd.concat(
        [
            structural_segment_distribution(family_train, group_cols, "train"),
            structural_segment_distribution(family_test, group_cols, "test"),
        ],
        ignore_index=True,
    ).round(4))

female_p3 = family_train.loc[family_train["Sex"].eq("female") & family_train["Pclass"].eq(3)]
male_p1 = family_train.loc[family_train["Sex"].eq("male") & family_train["Pclass"].eq(1)]

display(structural_rate_table(female_p3, ["FamilySurnameBand"]).round(4))
display(structural_rate_table(male_p1, ["FamilySurnameBand"]).round(4))
display(structural_rate_table(family_train, ["Sex", "Pclass", "FamilySurnameBand"]).round(4))


,dataset,Sex,FamilySurnameBand,count,rate
0,train,female,match_alone,100,0.3185
1,train,female,match_large,5,0.0159
2,train,female,match_medium,15,0.0478
3,train,female,match_small,101,0.3217
4,train,female,mismatch_family_alone_surname_medium,6,0.0191
5,train,female,mismatch_family_alone_surname_small,20,0.0637
6,train,female,mismatch_family_large_surname_large,8,0.0255
7,train,female,mismatch_family_medium_surname_medium,3,0.0096
8,train,female,mismatch_family_medium_surname_small,2,0.0064
9,train,female,mismatch_family_small_surname_alone,16,0.0510


,dataset,Pclass,FamilySurnameBand,count,rate
0,train,1,match_alone,91,0.4213
1,train,1,match_medium,6,0.0278
2,train,1,match_small,63,0.2917
3,train,1,mismatch_family_alone_surname_medium,3,0.0139
4,train,1,mismatch_family_alone_surname_small,15,0.0694
5,train,1,mismatch_family_small_surname_alone,12,0.0556
6,train,1,mismatch_family_small_surname_medium,5,0.0231
7,train,1,mismatch_family_small_surname_small,21,0.0972
8,train,2,match_alone,88,0.4783
9,train,2,match_small,56,0.3043


,dataset,Sex,Pclass,FamilySurnameBand,count,rate
0,train,female,1,match_alone,27,0.2872
1,train,female,1,match_medium,4,0.0426
2,train,female,1,match_small,32,0.3404
3,train,female,1,mismatch_family_alone_surname_medium,1,0.0106
4,train,female,1,mismatch_family_alone_surname_small,6,0.0638
...,...,...,...,...,...,...
99,test,male,3,mismatch_family_medium_surname_medium,1,0.0068
100,test,male,3,mismatch_family_medium_surname_small,1,0.0068
101,test,male,3,mismatch_family_small_surname_large,2,0.0137
102,test,male,3,mismatch_family_small_surname_medium,1,0.0068


,FamilySurnameBand,count,survived_count,survival_rate
0,match_alone,48,30.0,0.6250
1,match_large,5,0.0,0.0000
2,match_medium,11,0.0,0.0000
3,match_small,40,23.0,0.5750
4,mismatch_family_alone_surname_medium,2,2.0,1.0000
5,mismatch_family_alone_surname_small,10,5.0,0.5000
6,mismatch_family_large_surname_large,8,3.0,0.3750
7,mismatch_family_medium_surname_medium,3,0.0,0.0000
8,mismatch_family_small_surname_alone,2,1.0,0.5000
9,mismatch_family_small_surname_medium,2,2.0,1.0000


,FamilySurnameBand,count,survived_count,survival_rate
0,match_alone,64,22.0,0.3438
1,match_medium,2,0.0,0.0000
2,match_small,31,14.0,0.4516
3,mismatch_family_alone_surname_medium,2,0.0,0.0000
4,mismatch_family_alone_surname_small,9,3.0,0.3333
5,mismatch_family_small_surname_alone,1,0.0,0.0000
6,mismatch_family_small_surname_medium,3,2.0,0.6667
7,mismatch_family_small_surname_small,10,4.0,0.4000


,Sex,Pclass,FamilySurnameBand,count,survived_count,survival_rate
0,female,1,match_alone,27,26.0,0.9630
1,female,1,match_medium,4,4.0,1.0000
2,female,1,match_small,32,30.0,0.9375
3,female,1,mismatch_family_alone_surname_medium,1,1.0,1.0000
4,female,1,mismatch_family_alone_surname_small,6,6.0,1.0000
5,female,1,mismatch_family_small_surname_alone,11,11.0,1.0000
6,female,1,mismatch_family_small_surname_medium,2,2.0,1.0000
7,female,1,mismatch_family_small_surname_small,11,11.0,1.0000
8,female,2,match_alone,25,22.0,0.8800
9,female,2,match_small,29,28.0,0.9655


### Step 17A Reading

`FamilyBand` mostly compresses `SibSp` and `Parch`, which are already present in `raw_tabular`. `SurnameCountBand` is not independent of `FamilySize`, but the mismatch rows are material in both train and test, so the useful structural novelty is concentrated in the divergence between same-ticket family composition and same-surname manifest structure. The safer next controlled check is one combined `FamilySurnameBand`, not separate independent `FamilyBand` and `SurnameCountBand` columns.

Recommendation for the next feature test: **C. test one combined FamilySurnameBand**. Boundary: no model check, no submission, no public-score tuning in this step.
